In [ ]:
import json
import os
import re
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import wandb
import xgboost as xgb
from dotenv import load_dotenv
from shapely.geometry import Point, box
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path(os.getcwd()).resolve()

while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")
DATABASE_URL = os.getenv("DATABASE_URL")

print(f"Project root: {PROJECT_ROOT}")
print(f"Database URL: {'Set' if DATABASE_URL else 'NOT SET'}")

## Load & Inspect Data Sources

In [ ]:
# Paths
SRTM_PATH = PROJECT_ROOT / "data" / "external" / "N03E101.hgt"
WATERWAYS_PATH = PROJECT_ROOT / "data" / "external" / "hotosm_mys_waterways_lines_geojson.geojson"
HOTSPOTS_PATH = PROJECT_ROOT / "ai" / "imputation" / "data" / "pj_hotspots.csv"
ACCESSIBILITY_PATH = PROJECT_ROOT / "routing" / "artifacts" / "cell_accessibility_handoff.csv"
FEATURES_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "imputation_features.csv"
MODEL_DIR = PROJECT_ROOT / "ai" / "imputation" / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "ai" / "imputation" / "artifacts"

for p in [SRTM_PATH, WATERWAYS_PATH, HOTSPOTS_PATH, ACCESSIBILITY_PATH]:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {p.relative_to(PROJECT_ROOT)}")

In [ ]:
# Raw SRTM .hgt format: Big-endian 16-bit signed integers
# Shape is either 1201x1201 (3-arcsec) or 3601x3601 (1-arcsec)
size = os.path.getsize(SRTM_PATH)
dim = int(np.sqrt(size / 2))

elevation_data = np.fromfile(SRTM_PATH, dtype=">i2").reshape((dim, dim))

# Parsing NXXEXXX filename for origin
match = re.search(r"([NS])(\d+)([EW])(\d+)", SRTM_PATH.name)
if match:
    ns, lat_deg, ew, lon_deg = match.groups()
    # N03 means south edge is 3, top edge is 4. Data is top-down.
    raster_y_origin = float(lat_deg) + 1.0 if ns == "N" else -float(lat_deg)
    raster_x_origin = float(lon_deg) if ew == "E" else -float(lon_deg)
else:
    raster_y_origin = 4.0
    raster_x_origin = 101.0

raster_dx = 1.0 / (dim - 1)
raster_dy = 1.0 / (dim - 1)
raster_rows, raster_cols = dim, dim

print(f"Elevation raster loaded: {SRTM_PATH.name} ({raster_rows}x{raster_cols})")
print(f"Origin: ({raster_x_origin}, {raster_y_origin})")
print(f"Extent: Lon=[{raster_x_origin}, {raster_x_origin + raster_cols * raster_dx:.2f}], "
      f"Lat=[{raster_y_origin - raster_rows * raster_dy:.2f}, {raster_y_origin}]")

In [ ]:
# Load PJ flood hotspots
hotspots_df = pd.read_csv(HOTSPOTS_PATH)
hotspots_gdf = gpd.GeoDataFrame(
    hotspots_df,
    geometry=gpd.points_from_xy(hotspots_df["lng"], hotspots_df["lat"]),
    crs="EPSG:4326",
)
print(f"Loaded {len(hotspots_gdf)} flood hotspot points")
hotspots_gdf

In [ ]:
# Load road density/accessibility data
accessibility_df = pd.read_csv(ACCESSIBILITY_PATH)
print(f"Loaded {len(accessibility_df)} cell accessibility rows")
print(f"Columns: {list(accessibility_df.columns)}")
accessibility_df.head()

In [ ]:
# Load waterways (spatial filter for features touching PJ)
# PJ bounding box with buffer
PJ_BBOX = box(101.58, 3.08, 101.70, 3.17)

print("Loading waterways (this may take a moment for the 60MB file)...")
waterways_gdf = gpd.read_file(str(WATERWAYS_PATH), bbox=(101.58, 3.08, 101.70, 3.17))
print(f"Loaded {len(waterways_gdf)} waterway features within PJ bbox")

keep_columns = ['waterway', 'geometry']
waterways_gdf = waterways_gdf[keep_columns]

if len(waterways_gdf) > 0:
    print(f"Waterway types: {waterways_gdf['waterway'].value_counts().to_dict() if 'waterway' in waterways_gdf.columns else 'N/A'}")
waterways_gdf.head()

In [ ]:
if DATABASE_URL:
    from sqlalchemy import create_engine
    engine = create_engine(DATABASE_URL)

    try:
        grid_cells_gdf = gpd.read_postgis("SELECT id, cell_id, geom, baseline_vulnerability FROM public.grid_cells", engine, geom_col="geom")
    except Exception:
        grid_cells_gdf = gpd.read_postgis("SELECT id, cell_id, geometry as geom, baseline_vulnerability FROM public.grid_cells", engine, geom_col="geom")

    try:
        shelters_gdf = gpd.read_postgis("SELECT id, name, geom FROM public.shelters", engine, geom_col="geom")
    except Exception:
        shelters_gdf = gpd.read_postgis("SELECT id, name, geometry as geom FROM public.shelters", engine, geom_col="geom")

    print(f"Loaded {len(grid_cells_gdf)} grid cells and {len(shelters_gdf)} shelters from DB")
else:
    print("DATABASE_URL not set. Using local CSVs.")
    grid_cells_gdf = None
    shelters_gdf = None


## 2. Feature Extraction

In [ ]:
def get_elevation_at_point(lon: float, lat: float) -> float:
    """Sample elevation value at a given lon/lat from the SRTM raster."""
    col = int((lon - raster_x_origin) / raster_dx)
    row = int((raster_y_origin - lat) / raster_dy)
    if 0 <= row < raster_rows and 0 <= col < raster_cols:
        return float(elevation_data[row, col])
    return np.nan


def get_slope_at_point(lon: float, lat: float) -> float:
    """Compute slope (degrees) at a point using 3x3 neighborhood gradient."""
    col = int((lon - raster_x_origin) / raster_dx)
    row = int((raster_y_origin - lat) / raster_dy)
    if 1 <= row < raster_rows - 1 and 1 <= col < raster_cols - 1:
        # Approximate cell size in meters at this latitude
        cell_size_m = raster_dx * 111320 * np.cos(np.radians(lat))
        # Sobel-like gradient
        dz_dx = (
            float(elevation_data[row - 1, col + 1]) + 2 * float(elevation_data[row, col + 1]) + float(elevation_data[row + 1, col + 1])
            - float(elevation_data[row - 1, col - 1]) - 2 * float(elevation_data[row, col - 1]) - float(elevation_data[row + 1, col - 1])
        ) / (8 * cell_size_m)
        dz_dy = (
            float(elevation_data[row + 1, col - 1]) + 2 * float(elevation_data[row + 1, col]) + float(elevation_data[row + 1, col + 1])
            - float(elevation_data[row - 1, col - 1]) - 2 * float(elevation_data[row - 1, col]) - float(elevation_data[row - 1, col + 1])
        ) / (8 * cell_size_m)
        slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
        return float(np.degrees(slope_rad))
    return np.nan


def get_min_distance_to_waterways(point: Point, waterways: gpd.GeoDataFrame) -> float:
    """Compute minimum distance (meters) from a point to nearest waterway."""
    if waterways.empty:
        return np.nan
    # Project to a local CRS for distance in meters (UTM zone 47N for PJ)
    point_gdf = gpd.GeoDataFrame(geometry=[point], crs="EPSG:4326").to_crs("EPSG:32647")
    waterways_proj = waterways.to_crs("EPSG:32647")
    distances = waterways_proj.geometry.distance(point_gdf.geometry.iloc[0])
    return float(distances.min())


def get_min_distance_to_hotspots(point: Point, hotspots: gpd.GeoDataFrame) -> float:
    """Compute minimum distance (meters) from a point to nearest flood hotspot."""
    if hotspots.empty:
        return np.nan
    point_gdf = gpd.GeoDataFrame(geometry=[point], crs="EPSG:4326").to_crs("EPSG:32647")
    hotspots_proj = hotspots.to_crs("EPSG:32647")
    distances = hotspots_proj.geometry.distance(point_gdf.geometry.iloc[0])
    return float(distances.min())

In [ ]:
# Build feature table, one row per cell from accessibility data
# Use cell centroids from the PJ 500m grid

if grid_cells_gdf is None:
    raise ValueError("grid_cells_gdf is missing. Ensure DATABASE_URL is set in .env")

# Grid parameters (from generate_grid.py assumptions)
# PJ bbox and 500m grid. We use accessibility cell_ids as our reference.
cell_ids = accessibility_df["cell_id"].tolist()

# Centroids in UTM (EPSG:32647) for metric accurate distance calculations
centroids = grid_cells_gdf.set_index("id").geometry.centroid

# Lookup map
cell_centroids = {
    cid: (centroids.loc[cid].x, centroids.loc[cid].y)
    for cid in cell_ids
    if cid in centroids.index
}

print(f"Cell centroids computed for {len(cell_centroids)} cells")

In [ ]:
from shapely.ops import unary_union

# Preproject for faster distance computation
print("Projecting waterways to UTM for distance calculations...")
waterways_utm = waterways_gdf.to_crs("EPSG:32647")

# Build a single unioned geometry for faster nearest point queries
waterways_union = unary_union(waterways_utm.geometry)

# Preproject hotspots
hotspots_utm = hotspots_gdf.to_crs("EPSG:32647")
hotspots_union = unary_union(hotspots_utm.geometry)

print("Projections ready.")

In [ ]:
# Extract features for all cells
print(f"Extracting features for {len(cell_centroids)} cells...")

# Merge accessibility data
acc_lookup = accessibility_df.set_index("cell_id")

records = []
for i, (cid, (lon, lat)) in enumerate(cell_centroids.items()):
    if (i + 1) % 50 == 0:
        print(f"  Processing cell {i + 1}/{len(cell_centroids)}...")

    elev = get_elevation_at_point(lon, lat)
    slope = get_slope_at_point(lon, lat)

    pt_utm = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs="EPSG:4326").to_crs("EPSG:32647").geometry.iloc[0]
    dist_river = pt_utm.distance(waterways_union)

    dist_hotspot = pt_utm.distance(hotspots_union)

    road_density = acc_lookup.loc[cid, "avg_road_density"] if cid in acc_lookup.index else 0.0

    travel_time = acc_lookup.loc[cid, "avg_travel_time_to_shelter_seconds"] if cid in acc_lookup.index else np.nan

    records.append({
        "cell_id": cid,
        "centroid_lon": lon,
        "centroid_lat": lat,
        "mean_elevation": elev,
        "mean_slope": slope,
        "dist_to_river_m": dist_river,
        "dist_to_hotspot_m": dist_hotspot,
        "road_density": road_density,
        "travel_time_to_shelter_s": travel_time,
    })

features_df = pd.DataFrame(records)
print(f"\nFeature extraction complete. Shape: {features_df.shape}")
features_df.describe()

In [ ]:
print("NaN counts:")
print(features_df.isna().sum())
print(f"\nTotal rows: {len(features_df)}")

# Drop rows with NaN elevation (cells outside raster)
features_clean = features_df.dropna(subset=["mean_elevation", "mean_slope"]).copy()
print(f"Rows after dropping NaN elevation/slope: {len(features_clean)}")

## Proxy Label Construction

Since real flood damage labels are unavailable, we construct a proxy (not ground truth damage data):

```
vulnerability = w1 * hotspot_proximity + w2 * low_elevation + w3 * river_proximity
```


In [ ]:
# Construct proxy vulnerability label
HOTSPOT_BUFFER_M = 500  # cells within 500m of hotspot get max hotspot_proximity
W_HOTSPOT = 0.50
W_ELEVATION = 0.25
W_RIVER = 0.25

df = features_clean.copy()

# Hotspot proximity: 1.0 if within buffer, decays linearly to 0 at 2km
HOTSPOT_MAX_DIST_M = 2000
df["hotspot_proximity"] = np.clip(
    1.0 - (df["dist_to_hotspot_m"] - HOTSPOT_BUFFER_M) / (HOTSPOT_MAX_DIST_M - HOTSPOT_BUFFER_M),
    0, 1
)
# Cells within buffer get max score
df.loc[df["dist_to_hotspot_m"] <= HOTSPOT_BUFFER_M, "hotspot_proximity"] = 1.0

# Low elevation: invert and normalize (lower = higher score, more flood prone)
elev_min = df["mean_elevation"].min()
elev_max = df["mean_elevation"].max()
df["low_elevation"] = 1.0 - (df["mean_elevation"] - elev_min) / (elev_max - elev_min + 1e-8)

# River proximity: closer = higher score (more flood prone), decay over 1km
RIVER_MAX_DIST_M = 1000
df["river_proximity"] = np.clip(1.0 - df["dist_to_river_m"] / RIVER_MAX_DIST_M, 0, 1)

# Weighted combination
df["vulnerability_label"] = (
    W_HOTSPOT * df["hotspot_proximity"]
    + W_ELEVATION * df["low_elevation"]
    + W_RIVER * df["river_proximity"]
)
df["vulnerability_label"] = df["vulnerability_label"].clip(0, 1)

print("Vulnerability label stats:")
print(df["vulnerability_label"].describe())
print("\nLabel distribution (5 bins):")
print(pd.cut(df["vulnerability_label"], bins=5).value_counts().sort_index())

In [ ]:
# Save features + label CSV
df.to_csv(FEATURES_OUTPUT_PATH, index=False)
print(f"Saved features to {FEATURES_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shape: {df.shape}")
df.head(10)

## XGBoost Training with Wandb Tracking

In [ ]:
FEATURE_COLS = [
    "mean_elevation",
    "mean_slope",
    "dist_to_river_m",
    "dist_to_hotspot_m",
    "road_density",
    "travel_time_to_shelter_s",
]
TARGET_COL = "vulnerability_label"

df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0)

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
print(f"Feature columns: {FEATURE_COLS}")

In [ ]:
run = wandb.init(
    project="disaster-readiness-imputation",
    name="xgboost-vulnerability-v1",
    config={
        "model": "XGBRegressor",
        "target": "vulnerability_score (per grid cell)",
        "features": FEATURE_COLS,
        "n_features": len(FEATURE_COLS),
        "n_samples_train": X_train.shape[0],
        "n_samples_test": X_test.shape[0],
        "label_weights": {"hotspot": W_HOTSPOT, "elevation": W_ELEVATION, "river": W_RIVER},
        "hotspot_buffer_m": HOTSPOT_BUFFER_M,
        "proxy_label": True,
        # XGBoost hyperparameters
        "n_estimators": 200,
        "max_depth": 5,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "min_child_weight": 3,
        "random_state": 42,
    },
)

config = wandb.config
print(f"wandb run: {run.url}")

In [ ]:
# Train XGBoost
model = xgb.XGBRegressor(
    n_estimators=config.n_estimators,
    max_depth=config.max_depth,
    learning_rate=config.learning_rate,
    subsample=config.subsample,
    colsample_bytree=config.colsample_bytree,
    reg_alpha=config.reg_alpha,
    reg_lambda=config.reg_lambda,
    min_child_weight=config.min_child_weight,
    random_state=config.random_state,
    objective="reg:squarederror",
    eval_metric="rmse",
)

# Fit with eval set for tracking
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False,
)

results = model.evals_result()
for epoch in range(len(results["validation_0"]["rmse"])):
    wandb.log({
        "epoch": epoch,
        "train_rmse": results["validation_0"]["rmse"][epoch],
        "test_rmse": results["validation_1"]["rmse"][epoch],
    })

print("Training complete.")
print(f"Final train RMSE: {results['validation_0']['rmse'][-1]:.4f}")
print(f"Final test RMSE: {results['validation_1']['rmse'][-1]:.4f}")

## Evaluation & Feature Importance

In [ ]:
y_pred = model.predict(X_test)
y_pred_clipped = np.clip(y_pred, 0, 1)

# Metrics
mae = mean_absolute_error(y_test, y_pred_clipped)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_clipped))
r2 = r2_score(y_test, y_pred_clipped)

print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²:   {r2:.4f}")

wandb.log({
    "test_mae": mae,
    "test_rmse_final": rmse,
    "test_r2": r2,
})

print("\nPrediction stats:")
print(f"  Min: {y_pred_clipped.min():.4f}")
print(f"  Max: {y_pred_clipped.max():.4f}")
print(f"  Mean: {y_pred_clipped.mean():.4f}")

In [ ]:
# Feature importance
importance = model.get_booster().get_score(importance_type="gain")
# Map f0, f1, ... back to feature names
importance_named = {
    FEATURE_COLS[int(k.replace("f", ""))]: v
    for k, v in importance.items()
}

importance_sorted = dict(sorted(importance_named.items(), key=lambda x: x[1], reverse=True))

print("Feature importance (gain):")
for feat, score in importance_sorted.items():
    print(f"  {feat}: {score:.4f}")

wandb.log({"feature_importance": importance_sorted})

wandb.log({
    "feature_importance_chart": wandb.plot.bar(
        wandb.Table(
            data=[[k, v] for k, v in importance_sorted.items()],
            columns=["feature", "importance_gain"],
        ),
        "feature",
        "importance_gain",
        title="Feature Importance (Gain)",
    )
})

In [ ]:
# Predict on all cells (not just test set) for full vulnerability map
y_all_pred = np.clip(model.predict(X), 0, 1)
df["predicted_vulnerability"] = y_all_pred

print("Full prediction stats:")
print(df["predicted_vulnerability"].describe())

print("\nTop 10 most vulnerable cells:")
df.nlargest(10, "predicted_vulnerability")[["cell_id", "centroid_lon", "centroid_lat", "mean_elevation", "dist_to_river_m", "dist_to_hotspot_m", "predicted_vulnerability"]]

## Export Model Artifacts

In [ ]:
import joblib

model_path = MODEL_DIR / "vulnerability_model.joblib"
joblib.dump(model, model_path)
print(f"Model saved to {model_path.relative_to(PROJECT_ROOT)}")

importance_path = ARTIFACTS_DIR / "feature_importance.json"
with open(importance_path, "w") as f:
    json.dump(importance_sorted, f, indent=2)
print(f"Feature importance saved to {importance_path.relative_to(PROJECT_ROOT)}")

metrics = {
    "model": "XGBRegressor",
    "model_version": "v1",
    "n_samples_train": int(X_train.shape[0]),
    "n_samples_test": int(X_test.shape[0]),
    "n_features": len(FEATURE_COLS),
    "features": FEATURE_COLS,
    "test_mae": round(float(mae), 4),
    "test_rmse": round(float(rmse), 4),
    "test_r2": round(float(r2), 4),
    "proxy_label": True,
    "proxy_label_weights": {"hotspot": W_HOTSPOT, "elevation": W_ELEVATION, "river": W_RIVER},
    "limitations": [
        "Proxy label constructed from hotspot proximity, elevation, and river proximity (not real damage data)",
        "SRTM 30m resolution limits elevation accuracy in urban areas",
        "Flood hotspot locations are approximate geocodes from JPS reports",
        "Model trained on ~460 grid cells for Petaling Jaya MVP area only",
    ],
}
metrics_path = ARTIFACTS_DIR / "training_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Training metrics saved to {metrics_path.relative_to(PROJECT_ROOT)}")

artifact = wandb.Artifact("vulnerability-model", type="model")
artifact.add_file(str(model_path))
artifact.add_file(str(importance_path))
artifact.add_file(str(metrics_path))
wandb.log_artifact(artifact)
print("Artifacts logged to wandb.")

In [ ]:
# Save predictions csv for db writeback
predictions_path = ARTIFACTS_DIR / "cell_predictions.csv"
df[["cell_id", "predicted_vulnerability"]].to_csv(predictions_path, index=False)
print(f"Predictions saved to {predictions_path.relative_to(PROJECT_ROOT)}")
print("These can be written to grid_cells.baseline_vulnerability via DB writeback script.")

In [ ]:
wandb.finish()
print("wandb run finished. Check dashboard for full results.")

**Outputs produced:**
- `data/processed/imputation_features.csv` — full feature table with proxy labels
- `ai/imputation/models/vulnerability_model.joblib` — trained XGBoost model
- `ai/imputation/artifacts/feature_importance.json` — feature importance scores
- `ai/imputation/artifacts/training_metrics.json` — evaluation metrics + limitations
- `ai/imputation/artifacts/cell_predictions.csv` — predictions for DB writeback

**Next:**
1. Run `scripts/run_imputation_model.py --write-db --verify` to write predictions to DB and verify
2. Verify with map visualization: `scripts/visualize_results.py`